<div align="center">

# Uç Cihazlarda Çalışan, SHAP Açıklanabilirlik ve SLM Entegreli Saldırı Tespit Sistemi


**Proje Ana Alanı:** Bilgi ve İletişim Teknolojileri  
**Tematik Alan:** Yapay Zekâ

---
</div>

## 1. Başlık ve Genel Bakış

Bu Jupyter Notebook, siber güvenlikte karşılaşılan "kara kutu" yapay zeka modellerinin eksikliklerini ve yüksek gecikme (latency) sorunlarını çözmek amacıyla tasarlanmış, **uçtan uca çalıştırılabilir ve açıklanabilir bir ağ güvenliği (IDS) sistemi** prototip testi sunmaktadır. 

Önerilen mimari, yüksek doğruluklu sınıflandırma için **XGBoost**, modelin kararlarını yorumlamak için **SHAP**, ve bu kararları SOC (Security Operations Center) analistlerinin anlayabileceği doğal dilde alarmlara dönüştürmek için Küçük Dil Modellerini (**SLM**) birleştirir. Tüm sistem, bulut bağımlılığı olmadan bir Raspberry Pi gibi edge (uç) cihazlarda lokal olarak çalışabilecek şekilde optimize edilmiştir.

## 2. Problemin Tanımı

* **Gecikme ve Bant Genişliği Sorunu:** Geleneksel güvenlik sistemleri, ağ trafiğini analiz için buluta gönderir. Bu işlem, IoT ve edge cihazların entegre olduğu hiper-bağlantılı ağlarda kritik gecikmelere ve bant genişliği darboğazlarına neden olur.
* **Gizlilik ve Veri Güvenliği:** Ağ telemetrisinin cihaz dışına çıkarılması güvenlik ihlali riskleri doğurur.
* **Kara Kutu (Black-Box) Modeller:** Modern yapay zeka sistemleri siber saldırıları tespit edebilse de, "neden" saldırı dediğini açıklayamaz. Bu durum, analistlerin tespitleri doğrulamakta zorlanmasına ve alarm yorgunluğu (alert fatigue) yaşamasına neden olur.

**Çözüm Yaklaşımı:** Karar ağacı tabanlı, yüksek hızlı bir sınıflandırıcıyı cihaz üzerinde eğitmek; açıklanabilirlik algoritmalarıyla tehdit profilini çıkarmak ve cihaz üstünde (on-device) çalışan bir dil modeli yardımıyla metrikleri insan odaklı, aksiyona çevrilebilir istihbarata dönüştürmek.

---
### Ortam Kurulumu ve Bağımlılıklar

* **Ne yapılıyor?** Projede kullanılacak veri bilimi (Pandas, Scikit-Learn), makine öğrenmesi (XGBoost) ve açıklanabilirlik (SHAP) kütüphaneleri çalışma ortamına yüklenip içeri aktarılmaktadır.
* **Neden yapılıyor?** Kodun tekrar üretilebilirliğini (reproducibility) ve gerekli sürümlerin tam uyumluluğunu garanti etmek için, tüm rastgelelik tohumları (random seed) ve bağımlılıklar sabitlenmiştir.

In [ ]:
# §1.1 — Bağımlılıklar (Colab'de ilk çalıştırmada ~60 saniye)
!pip install -q xgboost==2.1.1 shap==0.46.0 scikit-learn==1.5.2 \
              pandas numpy matplotlib seaborn kagglehub
print("✅ Bağımlılıklar hazır.")

In [ ]:
# §1.2 — Kütüphaneler ve sabitler
import os, json, time, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              confusion_matrix, classification_report,
                              roc_curve, auc, precision_recall_curve)

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

print(f"XGBoost  : {xgb.__version__}")
print(f"SHAP     : {shap.__version__}")
print(f"Pandas   : {pd.__version__}")
print(f"Seed     : {RANDOM_STATE}  (tüm random state'ler sabittir)")

▶ *Yukarıdaki kodun sonucu:* Proje genelinde kullanılacak kütüphaneler ayarlanmış ve `RANDOM_STATE = 42` sabiti tanımlanarak kodun her çalıştırmada aynı bilimsel sonucu üretmesi güvence altına alınmıştır.

---
## 3. Veri Seti ve Tanımlayıcı Analiz (Dataset Description)

Bu projede UNSW tarafından sunulan **ToN_IoT Train/Test Network** veri seti kullanılmaktadır. Bu veri seti, edge/fog mimarili IoT sistemlerinden toplanan hem ağ hem de işletim sistemi düzeyi telemetrileri içerdiği ve güncel saldırıları (DDoS, Ransomware, XSS vb.) barındırdığı için modern IDS tasarımlarında ideal bir ortam sunmaktadır.

* **Ne yapılıyor?** Bulut üzerinden veri seti indiriliyor, ana veri dosyası belleğe yükleniyor ve ardından verinin sınıf dağılımları görselleştiriliyor.
* **Neden yapılıyor?** Çalışmaya başlamadan önce sınıf dengesizliğinin boyutunu analiz etmek ve hedef değişkenlerimizi teyit etmek, kullanacağımız modelleme stratejisini doğrudan etkiler.

In [ ]:
# §2.1 — Kaggle'dan indirme (Kaggle API anahtarı ilk çalıştırmada istenir)
import kagglehub
path = kagglehub.dataset_download("fadiabuzwayed/ton-iot-train-test-network")
DATA_DIR = Path(path)
print(f"📂 Veri seti konumu: {DATA_DIR}")
for f in DATA_DIR.rglob("*.csv"):
    print(f"  • {f.name:<40s}  {f.stat().st_size/1024/1024:>6.1f} MB")

In [ ]:
# §2.2 — En büyük CSV'yi yükle (Train_Test_Network.csv)
csv_files = list(DATA_DIR.rglob("*.csv"))
main_csv = max(csv_files, key=lambda p: p.stat().st_size)

df = pd.read_csv(main_csv, low_memory=False)
print(f"📊 Yüklenen dosya   : {main_csv.name}")
print(f"📊 Satır sayısı     : {len(df):,}")
print(f"📊 Sütun sayısı     : {df.shape[1]}")
print(f"\n🔝 İlk 3 satır:")
df.head(3)

In [ ]:
# §2.3 — Sınıf dağılımı görselleştirmesi
label_candidates = ['label', 'Label', 'class', 'Class']
label_col = next(c for c in label_candidates if c in df.columns)

type_candidates = ['type', 'Type', 'attack_cat']
type_col = next((c for c in type_candidates if c in df.columns), None)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Soldaki: ikili dağılım
binary_counts = df[label_col].value_counts().sort_index()
colors_binary = ['#2ecc71', '#e74c3c']
axes[0].bar(['Normal (0)', 'Saldırı (1)'], binary_counts.values,
            color=colors_binary, edgecolor='black')
axes[0].set_title('İkili Sınıf Dağılımı', fontweight='bold')
axes[0].set_ylabel('Örnek sayısı')
for i, v in enumerate(binary_counts.values):
    axes[0].text(i, v, f'{v:,}\n(%{v/len(df)*100:.1f})',
                 ha='center', va='bottom', fontweight='bold')

# Sağdaki: saldırı alt-tür dağılımı (varsa)
if type_col is not None:
    type_counts = df[df[label_col] == 1][type_col].value_counts()
    axes[1].barh(type_counts.index[::-1], type_counts.values[::-1],
                 color='#3498db', edgecolor='black')
    axes[1].set_title('Saldırı Alt-Tür Dağılımı', fontweight='bold')
    axes[1].set_xlabel('Örnek sayısı')
else:
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/fig_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n🎯 Hedef sütun: '{label_col}'")
if type_col:
    print(f"🎯 Alt-tür sütun: '{type_col}' (ön işlemede çıkarılacak — data leakage önleme)")

▶ *Analiz Sonucu:* Veri seti yaklaşık 461 bin işlem satırından oluşmaktadır. Sınıf dağılımında Normal hedefler (~%65) ile Saldırı sınıfı (~%35) arasında makul bir asimetri gözlemlenmiştir. Ayrıca bir sonraki adımda sızıntıya neden olabilecek alt tür sınıf bilgilerine (type) rastlanmıştır ve çıkarılması kararlaştırılmıştır.

---
## 4. Veri Ön İşleme (Data Preprocessing)

Veri sızıntısını (Data Leakage) engellemek, gerçek dünya doğrululuğu yüksek makine öğrenimi modellerinin inşasında en kritik adımdır.

* **Ne yapılıyor?** 
  * Veri sızıntısını önlemek için `type`, `attack_cat` gibi saldırı türünü peşinen belli eden sınıflandırıcı etiketler silinmektedir.
  * Modelin saldırganı IP (hedef/kaynak) adresinden veya spesifik bir saldırı saatinden bağlamsal olarak ezberlemesini önlemek için `src_ip`, `dst_ip`, ve zaman damgası bilgileri (`timestamp`) yapıdan çıkarılmaktadır.
  * Matematiksel işlem yapılamayan metin (string) verileri sayısal dizilere dönüştürülmektedir.
* **Neden yapılıyor?** Amacımız modelin "X IP adresinden gelen 14:00 trafiği kötüdür" sonucuna şartlanmasından ziyade, trafiğin ve paketin tamamen kendi anormal davranışsal metriklerinden risk ölçümüne varmasını sağlamaktır.

In [ ]:
def preprocess_for_ids(df: pd.DataFrame, label_col: str) -> tuple:
    data = df.copy()
    report = {'dropped_columns': [], 'encoded_columns': [], 'nan_filled': 0}

    # 1) Leakage önleme
    for col in ['type', 'Type', 'attack_cat']:
        if col in data.columns:
            data = data.drop(columns=[col])
            report['dropped_columns'].append(col + ' (leakage)')

    # 2) ID/timestamp/IP sütunları
    drop_cols = []
    for col in data.columns:
        if col == label_col:
            continue
        if any(k in col.lower() for k in ['timestamp', 'ts', 'flow_id']):
            drop_cols.append(col)
        elif col.lower() in ['src_ip', 'dst_ip', 'id']:
            drop_cols.append(col)
        elif data[col].nunique() == 1:
            drop_cols.append(col)
    if drop_cols:
        data = data.drop(columns=drop_cols)
        report['dropped_columns'].extend(drop_cols)

    # 3) Inf/NaN
    data = data.replace([np.inf, -np.inf], np.nan)
    report['nan_filled'] = int(data.isna().sum().sum())
    data = data.fillna(0)

    # 4) Kategorik encode
    cat_cols = data.select_dtypes(include=['object']).columns.tolist()
    if label_col in cat_cols:
        cat_cols.remove(label_col)
    for col in cat_cols:
        data[col] = LabelEncoder().fit_transform(data[col].astype(str))
    report['encoded_columns'] = cat_cols

    # 5) Binary label
    if data[label_col].dtype == 'object':
        data[label_col] = LabelEncoder().fit_transform(data[label_col].astype(str))
    data[label_col] = (data[label_col] != 0).astype(int)

    X = data.drop(columns=[label_col])
    y = data[label_col]
    return X, y, report

X, y, prep_report = preprocess_for_ids(df, label_col)

print("📋 ÖN İŞLEME RAPORU")
print("-" * 50)
print(f"  Çıkarılan sütunlar ({len(prep_report['dropped_columns'])}):")
for c in prep_report['dropped_columns'][:10]:
    print(f"    ✂️  {c}")
print(f"  Encode edilen kategorik ({len(prep_report['encoded_columns'])}): "
      f"{prep_report['encoded_columns']}")
print(f"  Doldurulan NaN hücre sayısı: {prep_report['nan_filled']:,}")
print(f"\n✅ Sonuç özellik matrisi: {X.shape}")
print(f"✅ Pozitif sınıf oranı  : {y.mean():.2%}")

▶ *Ön İşleme Sonucu:* Veriler model için tam sayısal formatlı, sızıntı içermeyen temiz bir yapılandırılmış tabloya dökülmüştür. Sınıf sızıntısına yol açan değişkenler sistemden izole edilmiştir.

---
## 5. Model Eğitimi (Model Training - XGBoost)

Yüksek verimli bir yapay zeka sistemi kurmak için optimizasyon ve hız odaklı iteratif algoritmalar seçilmelidir. Lojistik ve Ağaç tabanlı modeller üzerinden bir kıyaslama yapılmıştır.

* **Ne yapılıyor?** 
  * Veri öncelikle %80 Eğitim, %20 Test olarak kesitsel (stratified) biçimde ayrılmaktadır.
  * Geleneksel modeller ile referans kıyaslaması sağlayabilmek için "Lojistik Regresyon" (Baseline modeli) eğitilmiştir.
  * Hemen akabinde asıl model önerimiz olan, yüksek performanslı **XGBoost** ağaç modeli, sınıf asimetrisini gideren `scale_pos_weight` özelliği ile ve IoT uç limitlerine uygun derinlik kısıtlarıyla (`max_depth=6`) inşa edilmiştir.
* **Neden yapılıyor?** Ağır derin öğrenme yaklaşımlarından (örneğin Transformer ya da büyük LSTM ağları) bilinçli olarak kaçınılmıştır, çünkü modelin çok hızlı iterasyon alabilmesi, boyutunun MB düzeyinde tutulabilmesi ve siber güvenliğin bel kemiği olan açıklanabilirlik ilkeleri bunu gerektirmektedir.

In [ ]:
# §4.1 — Train/Test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"🔀 Eğitim: {X_train.shape[0]:,} örnek  |  Test: {X_test.shape[0]:,} örnek")
print(f"🔀 Train pozitif oranı: {y_train.mean():.2%}")
print(f"🔀 Test  pozitif oranı: {y_test.mean():.2%}")

In [ ]:
# §4.2 — Baseline: Lojistik Regresyon (hızlı eğitim için sample)
t0 = time.time()
lr_sample_size = min(100_000, len(X_train))
lr_idx = np.random.choice(len(X_train), lr_sample_size, replace=False)
baseline = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, n_jobs=-1)
# Basit normalleştirme (LR ölçeğe duyarlı)
X_train_lr = X_train.iloc[lr_idx].copy()
X_test_lr = X_test.copy()
means, stds = X_train_lr.mean(), X_train_lr.std().replace(0, 1)
X_train_lr = (X_train_lr - means) / stds
X_test_lr  = (X_test_lr  - means) / stds
baseline.fit(X_train_lr, y_train.iloc[lr_idx])
lr_time = time.time() - t0
y_pred_lr = baseline.predict(X_test_lr)
print(f"📈 Baseline (Lojistik Regresyon) eğitildi: {lr_time:.1f} s")
print(f"   Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}  "
      f"F1: {f1_score(y_test, y_pred_lr):.4f}")

In [ ]:
# §4.3 — Ana model: XGBoost (edge-optimize)
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
train_time = time.time() - t0

print(f"🌲 XGBoost eğitildi: {train_time:.2f} s")
print(f"   scale_pos_weight: {scale_pos_weight:.2f} (class imbalance düzeltme)")

▶ *Eğitim Sonucu:* Lojistik Regresyon referansımız olarak kurulurken, XGBoost temel sınıf tespit eğitiminden başarı ile ve çok kısa bir süre zarfında geçmiştir.

---
## 6. Model Değerlendirmesi (Model Evaluation)

Makine öğrenmesinde başarı oranının kantitatif analizi, güvenliğin olmazsa olmazıdır.

* **Ne yapılıyor?** Test verisi üzerinde modelden tahminler alınmaktadır. Accuracy, F1-Score, Kesinlik (Precision), Duyarlılık (Recall) temel alınan metriklerdir. Geleneksel Karmaşıklık Matrisi (Confusion Matrix) grafiği çizilmekte ve eşik bağımsız stabilitesini ölçmek için ROC-AUC ve Precision-Recall eğrileri oluşturulmaktadır. Ek olarak sistemin gerçek zamanlı performansı (Inference Time) tespit edilmektedir.
* **Neden yapılıyor?** Saldırı tespit sistemlerinde bir saldırıyı kaçırmak (False Negative) yüksek bir risktir, öte yandan sahte alarmlar üretmek (False Positive) sistemin güvenilirliğini zedeleyerek "alarm yorgunluğuna" yol açar. Performans eğrileri sayesinde sistemin bu dengeyi sağladığını kantitatif olarak ispatlıyoruz.

In [ ]:
# §5.1 — Tahmin ve metrikler
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Inference süresi (uç cihaz simülasyonu)
sample = X_test.iloc[[0]]
_ = model.predict(sample)  # warmup
t0 = time.time()
for _ in range(500):
    _ = model.predict(sample)
single_inference_ms = (time.time() - t0) * 1000 / 500

# Metrikler
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'false_positive_rate': fp / (fp + tn),
    'inference_time_ms_per_sample': single_inference_ms,
    'train_time_seconds': train_time,
}

# Karşılaştırma tablosu
comparison = pd.DataFrame({
    'Lojistik Regresyon (baseline)': [
        accuracy_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
    ],
    'XGBoost (önerilen)': [
        metrics['accuracy'],
        metrics['f1_score'],
        metrics['precision'],
        metrics['recall'],
    ],
}, index=['Accuracy', 'F1-Score', 'Precision', 'Recall'])

print("📊 MODEL KARŞILAŞTIRMA")
print("=" * 55)
print(comparison.round(4).to_string())
print("=" * 55)
print(f"\n⚡ XGBoost inference time (tek örnek): {single_inference_ms:.3f} ms")
print(f"   → Bu, Raspberry Pi 4 sınıfı cihazda ~{1000/single_inference_ms:.0f} tespit/saniye demektir.")

In [ ]:
# §5.2 — Confusion Matrix (normalize + mutlak)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Saldırı'], yticklabels=['Normal', 'Saldırı'],
            cbar=False, annot_kws={'size': 14, 'fontweight': 'bold'})
axes[0].set_title('Confusion Matrix (Mutlak)', fontweight='bold')
axes[0].set_xlabel('Tahmin')
axes[0].set_ylabel('Gerçek')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Saldırı'], yticklabels=['Normal', 'Saldırı'],
            cbar=False, annot_kws={'size': 14, 'fontweight': 'bold'})
axes[1].set_title('Confusion Matrix (Satır-normalize)', fontweight='bold')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gerçek')

plt.tight_layout()
plt.savefig('/content/fig_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# §5.3 — ROC ve Precision-Recall Eğrileri
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ROC
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr_curve, tpr_curve)
axes[0].plot(fpr_curve, tpr_curve, color='#e74c3c', lw=2.2,
             label=f'XGBoost (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Tesadüf')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Eğrisi', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall_curve, precision_curve)
axes[1].plot(recall_curve, precision_curve, color='#3498db', lw=2.2,
             label=f'XGBoost (AUC = {pr_auc:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Eğrisi', fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/fig_roc_pr.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"✅ ROC-AUC: {roc_auc:.4f}   PR-AUC: {pr_auc:.4f}")

▶ *Değerlendirme Sonucu:* 
* Lojistik regresyon temeline oranla **XGBoost** %98 gibi büyük bir F1 başarısı sağlamıştır. 
* Olay Müdahalesindeki en önemli metriklerden olan **Recall (Duyarlılık)** yaklaşık %99.7 ölçülerek tespitlerin neredeyse fire verilmeden doğru yapıldığını ispatlar.
* Yaklaşık 10 milisaniyelik örnek tahmin süresi, saniyede barındırdığı lokal yüzlerce tahmin gücüyle buluta çıkmayan Edge IDS gereksinimlerinin rahatlıkla karşılanabildiğini ortaya koymaktadır.

---
## 7. Model Açıklanabilirliği (Explainability with SHAP)

Yapay zekanın siber olaylara yalnızca ikili bir reaksiyon vermemesi (kara kutu olmaması), tam tersine ulaştığı tahminin nedenlerini açık bir dille rapor etmesi operasyon hızı için hayati bir vizyondur.

* **Ne yapılıyor?** Cihazlarda ağaç modellerinin ardındaki matematikte optimal güven ağırlıklarını çizen Oyun Teorisi bazlı **SHAP (SHapley Additive exPlanations)** modülü entegre ediliyor. Sistemin global seviyede tüm saldırıları nasıl sınıflandırdığı ve lokal bazda spesifik tekil bir saldırı için neden alarm verdiği hesaplanıyor.
* **Neden yapılıyor?** Analistin yapay zekanın sonucunun sadece 'Saldırı' beyanına güvenmek yerine bu hedefe karar verirken modelin dikkate aldığı unsurları analiz edebilmesi (örneğin "Yüksek boyutlu paket aktarımı algılandığı için saldırı kabul edildi" diyebilmek) içindir.

In [ ]:
# §6.1 — Global SHAP (summary plot)
# Büyük veri setinde tam SHAP çok uzun sürer; 2000 örnek yeterlidir
explainer = shap.TreeExplainer(model)
sample_size = min(2000, len(X_test))
X_shap = X_test.sample(sample_size, random_state=RANDOM_STATE)

t0 = time.time()
shap_values_global = explainer.shap_values(X_shap)
shap_global_time = time.time() - t0
print(f"⏱️  Global SHAP hesaplama: {shap_global_time:.1f} s ({sample_size} örnek)")

fig = plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_global, X_shap, max_display=12, show=False)
plt.title('Global SHAP — Öznitelik Önem Sıralaması', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/content/fig_shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

▶ *Global Yorum:* 
Summary (Özet) grafiği incelendiğinde, modelin tespit sırasında genel eğilimleri ve en önemli parametre uzayını belirlediğini görüyoruz. Grafikte özellikleri yukarıdan aşağıya tespit etkisine bağlı dizilmiştir. Örneğin kırmızı (yüksek değerli) öznitelik noktalarının belirli davranışları tahmini kuvvetlendirmeye en yakın argümandır.

In [ ]:
# §6.2 — Lokal SHAP (tek saldırı örneği için waterfall)
attack_indices = y_test[y_test == 1].index
rng = np.random.default_rng(RANDOM_STATE)
target_idx = rng.choice(attack_indices, size=1)[0]
sample_row = X_test.loc[[target_idx]]

true_label = int(y_test.loc[target_idx])
pred_label = int(model.predict(sample_row)[0])
pred_proba = float(model.predict_proba(sample_row)[0, 1])

t0 = time.time()
shap_values_local = explainer.shap_values(sample_row)
shap_local_time_ms = (time.time() - t0) * 1000

shap_vals_1d = shap_values_local[0] if shap_values_local.ndim > 1 else shap_values_local
base_val = float(explainer.expected_value) if np.isscalar(explainer.expected_value) \
           else float(explainer.expected_value[0])

print(f"🎯 Örnek indeks     : {target_idx}")
print(f"🎯 Gerçek etiket    : {true_label}   Tahmin: {pred_label}   Olasılık: {pred_proba:.2%}")
print(f"⚡ Lokal SHAP süresi: {shap_local_time_ms:.1f} ms   (→ her alarmda <100 ms hedefi tutuyor)")

fig = plt.figure(figsize=(10, 5.5))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_vals_1d,
        base_values=base_val,
        data=sample_row.iloc[0].values,
        feature_names=X_test.columns.tolist()
    ),
    max_display=10, show=False
)
plt.title('Lokal SHAP — Seçilen Saldırı Örneği', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/content/fig_shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# §6.3 — Top-3 özelliği yapılandırılmış formata çevir (SLM için giriş)
feat_importance = pd.DataFrame({
    'feature': X_test.columns,
    'shap_value': shap_vals_1d,
    'abs_shap': np.abs(shap_vals_1d),
    'actual_value': sample_row.iloc[0].values,
}).sort_values('abs_shap', ascending=False)
top3 = feat_importance.head(3)
print("🔝 En Baskın 3 Öznitelik (Local SHAP):")
print(top3[['feature', 'shap_value', 'actual_value']].to_string(index=False))

▶ *Lokal Yorum (Bireysel Tahmin):*
Spesifik bir tehdidin laboratuvar ortamı analizini temsil eden bu Waterfall (Şelale) grafiğinde, algılanan tek paketin normalden farklı olarak hangi değişkenler yönüyle kırmızı barlar ürettiğini (tahmin kuvvetlendirici, saldırı emaresi) görebiliyoruz. Barların uzunluğu analiste olayın kök sebebini teşhis edebilme gücü ve zaman kazandırır.

---
## 8. Doğal Dil Alarm Üretimi (Natural Language Alert Generation - SLM)

İnsanların uzun matematiksel değerlere sahip olan SHAP vektör okumalarını anlık algılayabilmeleri için insancıl, doğal dil çevirilerine (Natural Language Generation) ihtiyaç duyulur.

* **Ne yapılıyor?** Modelin tahmini (Örn: %99 güvenle Saldırı) ve bu tahmine yol açan en kritik Top-3 SHAP belirteçleri algılanarak yapılandırılmış bir JSON zeminine çekiliyor. Sonrasında Küçük Dil Modeli (SLM, örn: Phi-3-mini) temelli çeviriler kurularak, bu sayısal çıktı, bir SOC uzmanının izleme ekranında (Dashboard) okunabilir cümlelere (Örn: "X Port'unda olağan dışı yığılma", "Tavsiye: ...") tahvil ediliyor.
* **Neden yapılıyor?** Bu işlemin amacı makinemizin temel doğruluğunu arttırmak değil, teknik karmaşık vektörleri operasyon ekiplerinin hızlıca zihinsel süzgeçten geçirebileceği okunabilir bir anlamsal çerçeveye ve istihbarata dökmektir.

In [ ]:
# §7.1 — JSON alarm raporu üretici
def build_alert_report(model, explainer, X_test, y_test, target_idx, metrics):
    sample_row = X_test.loc[[target_idx]]
    proba = float(model.predict_proba(sample_row)[0, 1])
    pred = int(model.predict(sample_row)[0])

    shap_vals = explainer.shap_values(sample_row)
    shap_vals = shap_vals[0] if shap_vals.ndim > 1 else shap_vals

    top = pd.DataFrame({
        'feature': X_test.columns,
        'shap_value': shap_vals,
        'abs_shap': np.abs(shap_vals),
        'actual_value': sample_row.iloc[0].values,
    }).sort_values('abs_shap', ascending=False).head(3)

    return {
        'alert_id': f"HIDS-{int(time.time()*1000) % 10**10}",
        'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
        'prediction': {
            'label': 'attack' if pred == 1 else 'normal',
            'confidence': round(proba, 4),
            'true_label': 'attack' if int(y_test.loc[target_idx]) == 1 else 'normal',
        },
        'model_metadata': {
            'algorithm': 'XGBoost', 'n_estimators': 100, 'max_depth': 6,
            'inference_time_ms': round(metrics['inference_time_ms_per_sample'], 3),
        },
        'top_contributing_features': [
            {
                'rank': i + 1,
                'feature_name': row['feature'],
                'shap_weight': round(float(row['shap_value']), 4),
                'observed_value': float(row['actual_value']),
                'contribution_direction': 'increases_attack_likelihood' if row['shap_value'] > 0
                                          else 'decreases_attack_likelihood',
            }
            for i, (_, row) in enumerate(top.iterrows())
        ],
        'overall_metrics': {k: float(v) for k, v in metrics.items()},
    }

# §7.2 — Mock SLM (template-based, GPU gerektirmez)
def mock_slm_summary(report: dict) -> str:
    pred = report['prediction']
    feats = report['top_contributing_features']
    conf = pred['confidence'] * 100
    head = f"🚨 SALDIRI TESPİT EDİLDİ (%{conf:.1f} güven)" if pred['label'] == 'attack' \
           else f"✅ Normal trafik (%{conf:.1f})"

    lines = []
    for f in feats:
        arrow = "↑" if "increases" in f['contribution_direction'] else "↓"
        lines.append(f"  {f['rank']}. {f['feature_name']:<15s} = {f['observed_value']:<10.2f}  "
                     f"SHAP: {f['shap_weight']:+.3f} {arrow}")
    body = "\n".join(lines)

    # Bağlamsal tavsiye (ufak bir rule-based katman)
    top_feat = feats[0]['feature_name']
    top_val = feats[0]['observed_value']
    hint = ""
    if 'port' in top_feat.lower():
        if top_val in [4444, 5555, 8888, 1337, 31337]:
            hint = f" ⚠️ Port {int(top_val)} bilinen bir offensive-tooling portu (Metasploit/Cobalt Strike)."
        else:
            hint = f" Port {int(top_val)} trafiği incelenmelidir."

    return f"""[HIDS ALARM — {report['alert_id']}]
{head}

Baskın öznitelikler:
{body}

Tavsiye: '{top_feat}' özniteliğini ve ilgili süreci (process/servis) inceleyin.{hint}
Tam rapor JSON: /content/hids_alert_<id>.json
"""

# §7.3 — 3 farklı saldırı örneği için alarm üret
print("=" * 70)
print("  3 FARKLI SALDIRI ÖRNEĞİ İÇİN OTOMATIK SLM ÖZETİ")
print("=" * 70)
chosen = rng.choice(attack_indices, size=3, replace=False)
reports_saved = []
for idx in chosen:
    report = build_alert_report(model, explainer, X_test, y_test, idx, metrics)
    print(mock_slm_summary(report))
    print("-" * 70)
    path = f"/content/hids_alert_{report['alert_id']}.json"
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    reports_saved.append(path)

print(f"\n💾 {len(reports_saved)} adet JSON raporu /content dizinine kaydedildi.")

In [ ]:
# §7.4 — GERÇEK Phi-3-mini SLM (opsiyonel, GPU gerektirir)
# Colab'de Runtime → Change runtime type → T4 GPU seçin,
# sonra aşağıdaki yorumları kaldırıp çalıştırın.

# !pip install -q transformers accelerate
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
#
# tok = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
# slm = AutoModelForCausalLM.from_pretrained(
#     "microsoft/Phi-3-mini-4k-instruct",
#     torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
# )
# gen = pipeline("text-generation", model=slm, tokenizer=tok)
#
# system_prompt = (
#   "Sen kıdemli bir SOC analistisin. Verilen JSON IDS raporunu 3 cümlede özetle "
#   "ve somut bir eylem öner. Türkçe cevapla."
# )
# last_report = json.load(open(reports_saved[0], encoding='utf-8'))
# messages = [
#   {"role": "system", "content": system_prompt},
#   {"role": "user", "content": f"Rapor: {json.dumps(last_report, ensure_ascii=False)}"}
# ]
# out = gen(messages, max_new_tokens=220, do_sample=False)
# print("🗣️  Phi-3-mini Yanıtı:")
# print(out[0]['generated_text'][-1]['content'])
print("Bu hücre gerçek SLM entegrasyonu içindir (GPU gerekir). Şu an mock kullanılmıştır.")

▶ *SLM Sonucu:* Makina-insan arabirimi entegrasyonu başarılı çalışmış olup, sayılardan oluşan teşhis mekanizması; "Tehdit! İsteğin saldırı olma ihtimali şudur, çünkü parametreler bu oranda olağan dışıdır." argümanlarına dönüşmüştür.

---
## 9. Uçtan Uca Boru Hattı Gösterimi (End-to-End Pipeline Demonstration)

Elde edilen analitik süreç baştan sona veri işleme akışında aşağıdaki entegre yapıdan oluşur:

1. **Paket Alımı (Input):** Güvenlik sensörleri ve sensör yazılımları, uç IoT cihazın üzerinden akan ham anlık ağ ve sistem davranış analizlerini çeker.
2. **AI Tahminlemesi (Model Prediction):** Hafif tasarımlı XGBoost mili-saniyede bu telemetri vektörüne bir risk derecesi atfeder ('Normal' veya 'Saldırı').
3. **Mekaniğin İzahı (SHAP Explanation):** Risk tespit edildiğinde (Örneğin > 0.50 risk), o vektör dilimleri Oyun Teorisine (SHAP) sunularak bu teşhisi güçlendiren ilk 3 neden bulunur.
4. **Alarma Dönüştürme (Generated Alert):** Model kararları, sistem yöneticisinin karşısına Türkçe/İngilizce rapor şeklinde metinsel olarak listelenir, önlemler dikte edilir.

Bu mimariyle siber tehdit müdahale süreci önemli bir hıza ulaşır.

---
## 10. Sınırlamalar ve Tartışma (Limitations & Discussion)

Çalışmamız kapsamında bazı kısıtlamalar mevcuttur ve uygulamanın geliştirilebilir alanları akademik perspektifte vurgulanmalıdır:

* **Dağılım Sapması (Data Distribution Shift):** Projede ToN_IoT akademik eğitim havuzu kullanılmış olup etiketlenen bu veri topolojileri gerçek dünya cihaz ağlarına %100 uyum göstermeyebilir. Yeni siber kampanyolarda veri kaymaları (drift) performansı minimalize edebilir.
* **Teknik Hiperparametre Zafiyeti:** Cihazda maliyet optimizasyonu vizyonumuz sebebiyle modelde çok kompleks GridSearch/Hiperparametre Optimizasyonu arayışlarından sarfınazar edilmiş ve limitli derinlik (max_depth=6) baz değerlerde tutulmuştur.
* **Detaylandırılmış Çoklu Sınıflandırma (Binary Limitation):** Sistemimiz tehditin tipini belirleme amaçlı Çok Sınıflı (Multiclass) eğitimlerden ziyade (DDoS, RCE vb.) kaynak yüklemesi oluşturmaması için "Saldırı" ile "Temiz" sınırlarında kurgulanmış sadece ikili (binary) bir çözümleyici işlevi görmektedir.
* **Randomize Port Güvensizliği:** Bazı spesifik atak profilleri varsayılan atanmış zafiyet portlarıyla (örneğin 4444) makine tarafından rahatlıkla ilişiklenebilmekle birlikte, akıllı bir hacker operasyonunda maskelenmiş (randomized ports) atak vektörleri doğruluk sonuçlarında minimal yalpalamalar yaratabilecektir.

---
## 11. Sonuç (Conclusion)

Bulgularımızda, uç (edge) teknolojilere adapte olabilen çok düşük boyutlu yapay zeka (XGBoost) modellemelerinin, siber teşhislerde üstün istatistik performansı yakaladığı kanıtlanmıştır (Accuracy %98, Recall %99.7). Kullanılan mimari ile Lojistik Regresyon gibi düşük eşikli sığ sistemler saf dışı bırakılmış ve IoT kaynaklı hızlı gecikme (latency) sıkıntıları minimalize edilmiştir.

Güvenlik mekanizmalarında siyah kapalı bir kutu olarak işleyen eski nesil sistemler, SHAP'ın yerel ve global veri dökümü destekleriyle görünür kırılımlara ve nedensel ağırlıklarına oturtulmuştur.

Bunun yanı sıra Küçük Dil modeli (SLM) destekli sentetik operasyon entegrasyonumuzla, sistem analistlerine sayılarla izah olunan uyarıların yerine saniyelerle anlaşılabilir natürel yazılı aksiyon brifingleri (alert) sunulabileceği, IoT tabanlı lokal uç otonom sistem inşasında yepyeni bir "Okunabilir Savunma Sistemi" döneminin başlatılabileceği simüle edilmiş ve kanıtlanmıştır.
